# 5.1 - Domain Adaptation (YouTube, active learning)

Same adaptation as 5.0, but the shifted domain is real YouTube video instead of a synthetic corruption. yt-dlp downloads one clip per class, the frozen backbone turns the frames into features, active learning keeps the most informative clips, and Reptile adapts the head. Needs live YouTube access on Colab (URLs can die or hit bot-blocks).

## **Connect Colab**

In [1]:
import os, sys, subprocess
from getpass import getpass

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

REPO   = "silvergjeka22/Unified-Lifelong-Action-Learning"
BRANCH = "embeddings"
ROOT   = "/content/ulal"

token = os.environ.get("GITHUB_TOKEN") or getpass("GitHub token: ")
os.environ["GITHUB_TOKEN"] = token

if os.path.isdir(os.path.join(ROOT, ".git")):
    subprocess.run(["git", "-C", ROOT, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", ROOT, "reset", "--hard", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    "https://" + token + "@github.com/" + REPO + ".git", ROOT], check=True)

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

In [2]:
os.environ["KAGGLE_USERNAME"] = "colab"
os.environ["KAGGLE_KEY"]      = "PASTE_YOUR_KAGGLE_KEY"

## **Download DATASET**

Drive holds the cache, the head and the frozen backbone. The YouTube videos download live with yt-dlp, so no UCF video is needed here.

In [ ]:
from src.bootstrap import setup
setup(drive=True, dataset=False)

In [ ]:
!pip install -q yt-dlp

In [4]:
# imports
%run /content/ulal/src/imports.py
set_seed(cfg.SEED)

ULAL_ROOT : /content/ulal
Device    : cuda
GPU       : Tesla T4 (15.6 GB)
Classes   : base=10 t1=10 t2=10 t3=10 t4=10


42

## **Load Cache, Head, Backbone**

The base head is the model being adapted; the frozen ResNet backbone re-extracts features
from both clean and corrupted clips.

In [5]:
CACHE, LSTM_STATE, META = load_feature_cache(cfg.FEAT_CACHE)
class_to_idx = {cls_name: i for i, cls_name in enumerate(cfg.SELECTED_CLASSES)}

head = load_or_train_base_head(f"{cfg.CKPT_DIR}/head_base.pt", CACHE, LSTM_STATE, len(cfg.SELECTED_CLASSES), device)

cfg.RESNET50_PATH = f"{cfg.DRIVE_PROJECT}/resnet50/models/ResNet50_10C_groupsplit.pth"
backbone = load_frozen_backbone(device, ResNet50LSTMTeacher, cfg.RESNET50_PATH,
                                hidden_size=cfg.TEACHER_HIDDEN, num_classes=len(cfg.SELECTED_CLASSES),
                                dropout_p=cfg.DROPOUT_P, remap_fn=remap_teacher_checkpoint)
print("head + frozen backbone ready")

Loaded 9 tensors, 526 MB in RAM
  per clip: (16, 2048)
Loaded base head <- /content/drive/MyDrive/apai/checkpoints/head_base.pt
  Dropped 53 num_batches_tracked buffer(s).
head + frozen backbone ready


## **Download YouTube clips**

One video per class from cfg.YOUTUBE_CLIPS, N_CLIPS clips each, same transform as UCF101.

Colab is often bot-blocked by YouTube and some URLs die. If you get 0 clips: (1) put current working URLs in cfg.YOUTUBE_CLIPS, and (2) upload a cookies.txt exported from a logged-in browser to /content/cookies.txt (picked up automatically below). yt-dlp also accepts non-YouTube URLs such as a direct .mp4 or an archive.org link, which need no cookies.

In [ ]:
N_CLIPS = 40
YT_COOKIES = "/content/cookies.txt" if os.path.exists("/content/cookies.txt") else None
print(f"cookies: {YT_COOKIES}")

clips_by_class, skipped = download_and_extract(cfg.YOUTUBE_CLIPS, cfg.YT_RAW_DIR,
                                               class_to_idx, cfg.CLIP_LEN, N_CLIPS, cookies=YT_COOKIES)
yt_classes = list(clips_by_class.keys())
print(f"youtube classes: {yt_classes}")

In [ ]:
if len(yt_classes) == 0:
    raise RuntimeError(
        "No YouTube clips downloaded. Colab is likely bot-blocked or the URLs are dead. "
        "Put working URLs in cfg.YOUTUBE_CLIPS and upload /content/cookies.txt from a "
        "logged-in browser, or use direct .mp4 / archive.org URLs that need no cookies."
    )

## **Extract Features**

Run the frozen backbone over every YouTube clip, then split each class into an active-learning pool and a held-out test set.

In [ ]:
yt_clip_list = []
yt_label_list = []
for cls_name in yt_classes:
    for clip in clips_by_class[cls_name]:
        yt_clip_list.append(clip)
        yt_label_list.append(class_to_idx[cls_name])

yt_clips = torch.stack(yt_clip_list)
yt_labels = torch.tensor(yt_label_list, dtype=torch.long)
print(f"youtube clips {tuple(yt_clips.shape)}")

In [ ]:
yt_ds = TensorDataset(yt_clips, yt_labels)
yt_loader = DataLoader(yt_ds, batch_size=cfg.BATCH_SIZE)
yt_s, yt_y = extract_frame_features(backbone, yt_loader, device, cfg.BACKBONE_DIM, torch.float32)
print(f"youtube features {tuple(yt_s.shape)}")

In [ ]:
N_TEST = 10
pool_s_list, pool_y_list = [], []
test_s_list, test_y_list = [], []
for cls_name in yt_classes:
    idx = class_to_idx[cls_name]
    cls_s = yt_s[yt_y == idx]
    cls_y = yt_y[yt_y == idx]
    test_s_list.append(cls_s[:N_TEST])
    test_y_list.append(cls_y[:N_TEST])
    pool_s_list.append(cls_s[N_TEST:])
    pool_y_list.append(cls_y[N_TEST:])

yt_test_s = torch.cat(test_s_list)
yt_test_y = torch.cat(test_y_list)
yt_pool_s = torch.cat(pool_s_list)
yt_pool_y = torch.cat(pool_y_list)
print(f"pool {tuple(yt_pool_s.shape)}  test {tuple(yt_test_s.shape)}")

## **Domain Gap (before adaptation)**

The UCF-trained head on clean UCF vs YouTube. The gap is what active learning plus Reptile try to close.

In [ ]:
clean_test_s, clean_test_y = CACHE['t0_test']
clean_before = eval_head(head, clean_test_s, clean_test_y, device)
yt_before = eval_head(head, yt_test_s, yt_test_y, device)
print(f"before | clean UCF: {clean_before:.2%} | youtube: {yt_before:.2%} | gap: {clean_before - yt_before:.2%}")

In [ ]:
clean_emb, clean_ey = head_embeddings(head, CACHE, [0], device, "test")
yt_emb = head.embed(yt_test_s.to(device)).cpu()
clean_stats = compute_class_stats(clean_emb, clean_ey)
yt_stats = compute_class_stats(yt_emb, yt_test_y)
shifts = measure_shift(clean_stats, yt_stats, cfg.SELECTED_CLASSES)

## **Active Learning**

Score the YouTube pool by predictive entropy and keep the most informative clips per class.

In [ ]:
pool_by_class = {}
for cls_name in yt_classes:
    idx = class_to_idx[cls_name]
    pool_by_class[cls_name] = yt_pool_s[yt_pool_y == idx]

selected = select_top_k(pool_by_class, cfg.AL_STRATEGY, cfg.AL_BUDGET_PER_CLASS, head=head, device=device)

In [ ]:
al_list, al_ylist = [], []
for cls_name, idxs in selected.items():
    al_list.append(pool_by_class[cls_name][idxs])
    al_ylist.extend([class_to_idx[cls_name]] * len(idxs))

al_s = torch.cat(al_list)
al_y = torch.tensor(al_ylist, dtype=torch.long)
al_t = head.embed(al_s.to(device)).cpu()
print(f"AL-selected youtube clips: {tuple(al_s.shape)}")

## **Reptile Adaptation**

Reptile is the pragmatic choice here: a frozen backbone, a small head and only a few clips, where MAML's query-set machinery does not pay off. Clean UCF exemplars in the replay buffer keep the base classes from being forgotten.

In [ ]:
buffer = SmartReplayBuffer(max_per_class=15)
base_s, base_yb = CACHE['t0_train']
base_emb_all, _ = head_embeddings(head, CACHE, [0], device, "train")

for cls_name in cfg.SELECTED_CLASSES:
    idx = class_to_idx[cls_name]
    mask = base_yb == idx
    buffer.add_class(idx, base_s[mask], base_emb_all[mask], strategy="hard")

print(f"buffer: {len(buffer.s_store)} classes")

In [ ]:
import copy
set_seed(cfg.SEED)
adapted = copy.deepcopy(head)

reptile_adapt(adapted, al_s, al_t, al_y, buffer, device,
              episodes=cfg.ADAPT_EPISODES, inner_lr=cfg.ADAPT_INNER_LR,
              inner_steps=cfg.ADAPT_INNER_STEPS, epsilon=cfg.ADAPT_EPSILON,
              mse_weight=0.5, eval_s=yt_test_s, eval_y=yt_test_y)

## **After adaptation**

In [ ]:
clean_after = eval_head(adapted, clean_test_s, clean_test_y, device)
yt_after = eval_head(adapted, yt_test_s, yt_test_y, device)
print(f"youtube : {yt_before:.2%} -> {yt_after:.2%}  (gain {yt_after - yt_before:+.2%})")
print(f"clean   : {clean_before:.2%} -> {clean_after:.2%}  (change {clean_after - clean_before:+.2%})")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
groups = ["youtube", "clean UCF"]
before = [yt_before * 100, clean_before * 100]
after  = [yt_after * 100,  clean_after * 100]
x = np.arange(len(groups))

ax.bar(x - 0.2, before, 0.4, label="before")
ax.bar(x + 0.2, after,  0.4, label="after")
ax.set_xticks(x)
ax.set_xticklabels(groups)
ax.set_ylabel("accuracy (%)")
ax.set_title("YouTube domain adaptation: gain vs forgetting")
ax.legend()

plt.tight_layout()
plt.savefig(f"{cfg.RESULTS_DIR}/stage_da_youtube.png", dpi=150, bbox_inches="tight")
plt.show()

## **Save**

In [ ]:
payload = {
    "stage":         "domain_adapt_youtube",
    "youtube_classes": yt_classes,
    "n_clips":        N_CLIPS,
    "al_strategy":    cfg.AL_STRATEGY,
    "al_budget":      cfg.AL_BUDGET_PER_CLASS,
    "clean_before":   float(clean_before), "clean_after": float(clean_after),
    "yt_before":      float(yt_before),    "yt_after":    float(yt_after),
}
os.makedirs(cfg.RESULTS_DIR, exist_ok=True)
with open(f"{cfg.RESULTS_DIR}/stage_da_youtube.json", "w") as f:
    json.dump(payload, f, indent=2, default=float)

torch.save(adapted.state_dict(), f"{cfg.CKPT_DIR}/head_adapted_youtube.pt")
print(f"Saved -> {cfg.RESULTS_DIR}/stage_da_youtube.json")